# vLLM Tutorial 1: Installation and Basic Usage

## Overview

This notebook covers:

- What is vLLM and why use it

- Installation process

- Basic text generation

- Understanding key components

- Sampling parameters

---

## What is vLLM?

**vLLM** (Virtual Large Language Model) is a fast and memory-efficient inference engine for serving Large Language Models (LLMs).

### Key Benefits:

- **4x faster throughput** compared to traditional methods (HuggingFace Transformers)

- **4x less memory usage** through PagedAttention algorithm

- **Continuous batching** for better GPU utilization (85-95% vs 40-60%)

- **Easy integration** with HuggingFace models

- **Production-ready** with OpenAI-compatible API server

### Core Innovation: PagedAttention

Traditional inference stores KV cache (key-value pairs from attention) in contiguous memory blocks, leading to:

- Memory fragmentation

- Wasted space

- Limited concurrent requests

**PagedAttention** divides KV cache into blocks (pages), similar to virtual memory in operating systems:

- Non-contiguous memory allocation

- Dynamic growth

- Memory sharing between requests

- Up to **4x memory savings**

In [ ]:
Traditional:  [████████████░░░░░░░░]  ← Wasted space
PagedAttention: [████][████][████]    ← Efficient blocks

---

## Installation

### Prerequisites

- Python 3.8 or higher

- CUDA 11.8 or higher (for GPU support)

- Linux or macOS (Windows via WSL2)

### Method 1: Install via pip (Recommended for beginners)

In [ ]:
# Install vLLM
!conda install -n agents -c conda-forge vllm -y

# For specific CUDA version (example: CUDA 12.1)
# !pip install vllm

### Method 2: Install from source (Latest features)

In [ ]:
# Install from GitHub (latest development version)
# !pip install git+https://github.com/vllm-project/vllm.git

### Verify Installation

In [ ]:
# Check vLLM version
import vllm
print(f"vLLM version: {vllm.__version__}")

# Check CUDA availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

---

## Basic Usage: Your First vLLM Generation

Let's start with a simple example using a small model to understand the basics.

### Example 1: Simple Text Generation

In [ ]:
from vllm import LLM, SamplingParams

# Step 1: Initialize the model
# We use a small model (125M parameters) for quick testing
# This will download the model from HuggingFace if not cached
llm = LLM(
    model="facebook/opt-125m",  # Small OPT model from Meta
    trust_remote_code=True       # Required for some models
)

print("Model loaded successfully!")

### Understanding the LLM Object

The `LLM` class is the main entry point for vLLM inference:

In [ ]:
llm = LLM(
    model="model_name",              # HuggingFace model ID or local path
    tensor_parallel_size=1,          # Number of GPUs for tensor parallelism
    max_num_seqs=256,                # Max concurrent request
    max_model_len=2048,              # Max sequence length
    gpu_memory_utilization=0.90,     # GPU memory to use (0.0-1.0)
    trust_remote_code=True           # Allow custom model code
)

In [ ]:
# Step 2: Define your prompts
prompts = [
    "The capital of France is",
    "Artificial Intelligence is",
    "The meaning of life is"
]

# Step 3: Configure sampling parameters
sampling_params = SamplingParams(
    temperature=0.8,    # Controls randomness (0.0 = deterministic, 1.0 = more random)
    top_p=0.95,         # Nucleus sampling (consider top 95% probability mass)
    max_tokens=50       # Maximum tokens to generate per prompt
)

# Step 4: Generate responses
outputs = llm.generate(prompts, sampling_params)

# Step 5: Print results
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"Prompt: {prompt}")
    print(f"Generated: {generated_text}")
    print("-" * 80)

---

## Understanding Sampling Parameters

Sampling parameters control how the model generates text. Let's explore each one:

### 1. Temperature

**What it does**: Controls randomness in generation

- **Low (0.0 - 0.3)**: More deterministic, focused, repetitive

- **Medium (0.7 - 0.9)**: Balanced creativity and coherence

- **High (1.0+)**: More random, creative, but potentially incoherent

**How it works**: Divides logits (model predictions) by temperature before softmax

In [ ]:
# Demonstrate temperature effect
prompt = "Once upon a time in a distant galaxy"

temperatures = [0.0, 0.5, 1.0, 1.5]

for temp in temperatures:
    sampling_params = SamplingParams(
        temperature=temp,
        max_tokens=30
    )

    output = llm.generate([prompt], sampling_params)[0]
    print(f"Temperature {temp}:")
    print(output.outputs[0].text)
    print("-" * 80)

### 2. Top-p (Nucleus Sampling)

**What it does**: Samples from the smallest set of tokens whose cumulative probability ≥ p

- **top_p=0.9**: Consider tokens making up top 90% probability

- **top_p=1.0**: Consider all tokens

- **top_p=0.1**: Very conservative, only highest probability tokens

In [ ]:
**Example**:
Token probabilities: A(0.4), B(0.3), C(0.2), D(0.1)
top_p=0.7 → samples from {A, B}  (0.4 + 0.3 = 0.7)
top_p=0.9 → samples from {A, B, C}  (0.4 + 0.3 + 0.2 = 0.9)

In [ ]:
# Demonstrate top_p effect
prompt = "The best programming language is"

top_p_values = [0.5, 0.75, 0.95, 1.0]

for top_p in top_p_values:
    sampling_params = SamplingParams(
        temperature=0.8,
        top_p=top_p,
        max_tokens=30
    )

    output = llm.generate([prompt], sampling_params)[0]
    print(f"Top-p {top_p}:")
    print(output.outputs[0].text)
    print("-" * 80)

### 3. Top-k Sampling

**What it does**: Samples from top-k most probable tokens

- **top_k=50**: Consider only 50 most likely tokens

- **top_k=-1**: No limit (consider all tokens)

---

### 🔑 Key Difference: top_k vs top_p (For Beginners)

Both control which tokens the model can choose from, but in different ways:

#### **top_k: Fixed Number**

- Picks a **fixed number** of most likely tokens

- Simple and predictable

- Example: `top_k=50` → always considers exactly 50 tokens

In [ ]:
Token probabilities (ranked):
1. "is"    → 40%
2. "was"   → 25%
3. "will"  → 15%
4. "can"   → 10%
5. "might" → 5%
...
50. "xyz"  → 0.01%

top_k=3 → samples from {"is", "was", "will"} only

#### **top_p: Dynamic Cutoff**

- Picks tokens until their **cumulative probability** reaches p

- Number of tokens varies based on probability distribution

- Example: `top_p=0.90` → considers as many tokens needed to reach 90% probability

In [ ]:
Same token probabilities:
top_p=0.80 → samples from {"is", "was", "will"} (40+25+15 = 80%)
top_p=0.50 → samples from {"is", "was"} (40+25 = 65%, includes "was")

#### **When to Use Which?**

| Scenario | Recommended | Why |

|----------|-------------|-----|

| **Creative writing** | `top_p=0.9-0.95` | Adapts to context, allows variety |

| **Factual answers** | `top_k=1-10` or `top_p=0.1` | Stays focused on likely tokens |

| **Code generation** | `top_p=0.9` + `top_k=50` | Balanced creativity + constraint |

| **General chatbot** | `top_p=0.95` + `temperature=0.7` | Most common setting |

#### **Can You Use Both?**

**Yes!** They work together:

1. First, `top_k` filters to K tokens

2. Then, `top_p` further filters within those K tokens

3. Finally, `temperature` adjusts randomness

In [ ]:
# Example: Conservative but creative
SamplingParams(
    top_k=40,          # Consider top 40 tokens
    top_p=0.85,        # Then keep tokens until 85% probability
    temperature=0.8    # Add slight randomness
)

#### **Quick Rules of Thumb**

- **top_k alone**: Use for simple, predictable generation

- **top_p alone**: Use for adaptive, context-aware generation ✅ (Most popular)

- **Both together**: Use when you want fine control

- **Neither (top_k=-1, top_p=1.0)**: Pure temperature-based sampling (very random)

### 4. Max Tokens

**What it does**: Maximum number of tokens to generate

- Controls generation length

- Model may stop earlier if it generates an end-of-sequence token

In [ ]:
# Demonstrate max_tokens effect
prompt = "Write a short story:"

max_tokens_values = [20, 50, 100]

for max_tokens in max_tokens_values:
    sampling_params = SamplingParams(
        temperature=0.8,
        max_tokens=max_tokens
    )

    output = llm.generate([prompt], sampling_params)[0]
    print(f"Max tokens {max_tokens}:")
    print(output.outputs[0].text)
    print(f"Actual tokens generated: {len(output.outputs[0].token_ids)}")
    print("-" * 80)

---

## Complete SamplingParams Reference

Here's a comprehensive overview of all sampling parameters:

In [ ]:
from vllm import SamplingParams

# All available parameters
sampling_params = SamplingParams(
    # Randomness control
    temperature=0.8,           # 0.0 = deterministic, higher = more random
    top_p=0.95,                # Nucleus sampling threshold
    top_k=-1,                  # Top-k sampling (-1 = disabled)

    # Length control
    max_tokens=100,            # Maximum tokens to generate
    min_tokens=0,              # Minimum tokens to generate

    # Repetition control
    presence_penalty=0.0,      # Penalize tokens that appeared (0.0 to 2.0)
    frequency_penalty=0.0,     # Penalize frequent tokens (0.0 to 2.0)
    repetition_penalty=1.0,    # Penalize repetitions (1.0 = no penalty)

    # Special tokens
    stop=["\n\n", "END"],      # Stop sequences
    ignore_eos=False,          # Ignore end-of-sequence token

    # Multiple outputs
    n=1,                       # Number of completions per prompt
    best_of=1,                 # Generate best_of and return top n

    # Advanced
    use_beam_search=False,     # Use beam search instead of sampling
    length_penalty=1.0,        # Exponential length penalty for beam search
    early_stopping=False,      # Stop beam search when best_of complete

    # Token control
    skip_special_tokens=True,  # Skip special tokens in output
    spaces_between_special_tokens=True,  # Add spaces between special tokens
)

# Display configuration
print("Sampling Parameters Configuration:")
print(sampling_params)

---

## Generating Multiple Outputs

vLLM can generate multiple completions for each prompt efficiently.

In [ ]:
# Generate multiple outputs per prompt
prompt = "Write a creative tagline for a coffee shop:"

sampling_params = SamplingParams(
    temperature=0.9,
    top_p=0.95,
    max_tokens=20,
    n=5  # Generate 5 different completions
)

outputs = llm.generate([prompt], sampling_params)

print(f"Prompt: {prompt}\n")
for i, output_obj in enumerate(outputs[0].outputs, 1):
    print(f"Option {i}: {output_obj.text}")

In [ ]:
import time

# Create a batch of diverse prompts
prompts = [
    "Explain quantum computing in simple terms:",
    "Write a haiku about mountains:",
    "List 5 benefits of exercise:",
    "Describe the color blue to someone who can't see:",
    "What is the fibonacci sequence?"
]

sampling_params = SamplingParams(
    temperature=0.7,
    max_tokens=60
)

# Measure batch processing time
start_time = time.time()
outputs = llm.generate(prompts, sampling_params)
elapsed_time = time.time() - start_time

# Display results
for i, output in enumerate(outputs, 1):
    print(f"\n{'='*80}")
    print(f"Request {i}:")
    print(f"Prompt: {output.prompt}")
    print(f"Response: {output.outputs[0].text}")

print(f"\n{'='*80}")
print(f"Processed {len(prompts)} prompts in {elapsed_time:.2f} seconds")
print(f"Average time per prompt: {elapsed_time/len(prompts):.2f} seconds")

---

## Understanding Output Objects

Let's explore what information vLLM returns.

In [ ]:
# Generate a sample output
prompt = "The future of technology is"
sampling_params = SamplingParams(temperature=0.8, max_tokens=30)

outputs = llm.generate([prompt], sampling_params)
output = outputs[0]  # First (and only) prompt result

# Explore the RequestOutput object
print("RequestOutput attributes:")
print(f"  - request_id: {output.request_id}")
print(f"  - prompt: {output.prompt}")
print(f"  - prompt_token_ids (first 10): {output.prompt_token_ids[:10]}")
print(f"  - prompt_logprobs: {output.prompt_logprobs is not None}")
print(f"  - outputs: {len(output.outputs)} completion(s)")
print(f"  - finished: {output.finished}")

# Explore the CompletionOutput object
completion = output.outputs[0]
print("\nCompletionOutput attributes:")
print(f"  - index: {completion.index}")
print(f"  - text: {completion.text}")
print(f"  - token_ids: {completion.token_ids}")
print(f"  - cumulative_logprob: {completion.cumulative_logprob}")
print(f"  - logprobs: {completion.logprobs is not None}")
print(f"  - finish_reason: {completion.finish_reason}")

---

## Using Stop Sequences

Control when generation stops using custom stop sequences.

In [ ]:
# Example: Generate a numbered list but stop after 3 items
prompt = "List the planets in our solar system:\n1."

sampling_params = SamplingParams(
    temperature=0.7,
    max_tokens=200,
    stop=["\n4."]  # Stop when "4." appears
)

output = llm.generate([prompt], sampling_params)[0]
print("Prompt + Generation:")
print(prompt + output.outputs[0].text)
print(f"\nStopped due to: {output.outputs[0].finish_reason}")

---

## Best Practices for Beginners

### 1. Choose Appropriate Temperature

- **Factual tasks** (QA, summarization): 0.0 - 0.3

- **Balanced tasks** (chat, dialogue): 0.7 - 0.9

- **Creative tasks** (story writing, brainstorming): 1.0+

In [ ]:
### 2. Combine top_p and temperature
# Good for most tasks
SamplingParams(temperature=0.8, top_p=0.95)

In [ ]:
### 3. Use stop sequences for structured output
# Example: JSON generation
SamplingParams(stop=["}"], max_tokens=100)

### 4. Batch your requests

- vLLM is optimized for batch processing

- Group similar requests together for best performance

In [ ]:
### 5. Monitor GPU memory
import torch
print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"GPU memory reserved: {torch.cuda.memory_reserved()/1e9:.2f} GB")

---

## Common Issues and Solutions

In [ ]:
### Issue 1: Out of Memory (OOM)
**Solution**:
llm = LLM(
    model="model_name",
    gpu_memory_utilization=0.85,  # Reduce from 0.90
    max_num_seqs=64,              # Reduce concurrent requests
    max_model_len=2048            # Reduce max sequence length
)

### Issue 2: Slow first generation

**Explanation**: First generation includes model loading and CUDA kernel compilation

**Solution**: This is normal; subsequent generations will be much faster

In [ ]:
### Issue 3: Repetitive output
**Solution**:
SamplingParams(
    temperature=0.9,
    repetition_penalty=1.2,  # Penalize repetitions
    frequency_penalty=0.5    # Penalize frequent tokens
)

---

## Summary

In this notebook, you learned:

- ✅ How to install vLLM

- ✅ Basic text generation with `LLM` class

- ✅ Sampling parameters (temperature, top_p, top_k, max_tokens)

- ✅ Generating multiple outputs

- ✅ Batch processing

- ✅ Understanding output objects

- ✅ Best practices and troubleshooting

### Next Steps

Continue to the next notebook to learn about:

- Supported models and architectures

- Model serving options and deployment

- Advanced configuration parameters